# Unit 6: Supply Chains — QUBO for Nurse Scheduling

**Companion notebook for *What Quantum Computers Are Actually For***

We formulate a small nurse scheduling problem as a QUBO, convert it to an Ising
Hamiltonian, solve it with QAOA on a cloud [Quokka](https://www.quokkacomputing.com/),
and compare with classical brute-force.

**What you'll see:**
1. Define a toy scheduling problem (4 nurses, 4 shifts)
2. Formulate constraints as QUBO penalty terms
3. Convert QUBO → Ising Hamiltonian → QAOA circuit
4. Run on Quokka and check which schedules are feasible

In [ ]:
import json
import itertools
import numpy as np
import matplotlib.pyplot as plt

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. Define the scheduling problem

4 nurses, 4 shifts (Mon-day, Mon-night, Tue-day, Tue-night).

**Hard constraint:** Each shift has exactly 1 nurse.

**Soft constraint:** Nurses 0 and 1 prefer day shifts (penalty for night assignment).

Binary variables: $x_{n,s} = 1$ if nurse $n$ assigned to shift $s$.

In [ ]:
n_nurses = 4
n_shifts = 4
shift_names = ['Mon-day', 'Mon-night', 'Tue-day', 'Tue-night']

# For a minimal demo, we assign one nurse per shift (4 nurses, 4 shifts)
# This is equivalent to a permutation problem
# We encode it with 4 qubits: qubit i represents which nurse gets shift i
# Simplified: binary encoding where each shift gets a 0/1 choice

# Even simpler: 4 binary variables, x_i = nurse assignment for shift i
# But let's use the QUBO on a smaller model:
# 2 shifts, 2 nurses, 2 binary variables

print("Simplified model: 2 nurses (A, B), 2 shifts")
print("x0 = 1 means Nurse A takes Shift 0")
print("x1 = 1 means Nurse A takes Shift 1")
print("(If x_i = 0, Nurse B takes that shift)")
print()

# QUBO: minimise preferences
# Nurse A prefers Shift 0 (day), penalty for Shift 1 (night)
# Constraint: Nurse A can only take one shift: x0 + x1 = 1
# → penalty: P * (x0 + x1 - 1)^2 = P * (x0^2 + x1^2 + 2*x0*x1 - 2*x0 - 2*x1 + 1)
# Since x_i^2 = x_i (binary): P * (x0 + x1 + 2*x0*x1 - 2*x0 - 2*x1 + 1)
#                              = P * (2*x0*x1 - x0 - x1 + 1)
# Preference: nurse A prefers shift 0, so add cost for x1
# Total: P*(2*x0*x1 - x0 - x1 + 1) + pref*x1

P = 5.0      # constraint penalty weight
pref = 1.0   # preference cost

# QUBO matrix Q where cost = x^T Q x
# Q[0,0] = -P, Q[1,1] = -P + pref, Q[0,1] = Q[1,0] = P
Q = np.array([
    [-P,     P],
    [ P, -P + pref]
])

print("QUBO matrix Q:")
print(Q)

# Evaluate all 4 assignments
print("\nAll assignments:")
print(f"{'x0':>3} {'x1':>3} | {'Cost':>8} | {'Feasible':>8} | Schedule")
print("-" * 55)
for x0, x1 in itertools.product([0, 1], repeat=2):
    x = np.array([x0, x1])
    cost = x @ Q @ x + P  # add constant from constraint
    feasible = (x0 + x1 == 1)
    sched = f"A→Shift{0 if x0 else 1}, B→Shift{1 if x0 else 0}" if feasible else "infeasible"
    print(f"{x0:>3} {x1:>3} | {cost:>8.1f} | {'✓' if feasible else '✗':>8} | {sched}")

## 2. Convert to Ising and run QAOA

In [ ]:
# QUBO → Ising: x_i = (1 - Z_i) / 2
# The Ising Hamiltonian has ZZ, Z, and constant terms
# For our 2-qubit problem:

J01 = Q[0, 1] / 4  # ZZ coefficient
h0 = (-Q[0, 0] / 2 - Q[0, 1] / 4)  # Z0 coefficient  
h1 = (-Q[1, 1] / 2 - Q[0, 1] / 4)  # Z1 coefficient

print(f"Ising coefficients: J01={J01:.2f}, h0={h0:.2f}, h1={h1:.2f}")

# Build QAOA circuit
gamma = 0.8
beta = 0.4

qaoa = f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Superposition
h q[0];
h q[1];

// Problem unitary: ZZ interaction
cx q[0], q[1];
rz({2*gamma*J01:.6f}) q[1];
cx q[0], q[1];

// Problem unitary: Z terms
rz({2*gamma*h0:.6f}) q[0];
rz({2*gamma*h1:.6f}) q[1];

// Mixer
rx({2*beta:.6f}) q[0];
rx({2*beta:.6f}) q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

results = run_qasm(qaoa, shots=1024)

print("\nQAOA results:")
for outcome in sorted(results.keys()):
    x0, x1 = int(outcome[0]), int(outcome[1])
    feasible = (x0 + x1 == 1)
    print(f"  {outcome}: {results[outcome]:>4} counts {'✓ feasible' if feasible else '✗ infeasible'}")

## What's next?

- **Circuit walkthrough:** See Chapter 2 (Building QAOA)
- Scale up: add more nurses, more shifts, more constraint types
- Compare with classical simulated annealing
- [Unit 7 — Materials Science](../manuscript/07-materials-science.md): QPE on the Hubbard model